# Proportional variability – simulation study

Code for M. Schlemmer, *Proportional variability relative to lower abundances: An interpretable and efficient index of deep lows in population time series* (bioRxiv). The notebook computes CV, SDL, pv<sub>lo</sub>, PV<sub>lo</sub> and PV<sub>hi</sub> for simulated population dynamics and produces the numbers, tables and figure data of the paper. In the decline analyses the consecutive disparity index D (Fernández-Martínez et al. 2018) is included as an order-sensitive reference.

**Contents**
- *Self-tests*: efficient method versus pairwise definition; reproduction of Table 2 (example series).
- *Simulation* (Normal, Cauchy, Rare, Bimodal) and *summary* with the correlation with the average deviation from the mean (ADM): Section 3.2, Table 3, Figs. 2–3.
- *Additional analysis (Rare)* and *series length* (variance exponents): Section 3.2 and the limitations in Section 4.
- *Decline regime*: Cells A and B (exploration of the parameters, not reported in the paper), Cell C (final analysis, Section 3.3, Tables 4–5), Cell D (crash versus ordinary fluctuations, log-normal variant, not used in the paper), Cell E (data for Fig. 4), Cell F (controls with known and unknown baseline amplitude, Table 6).
- *Cauchy events* (Cell G): frequency of booms and crashes in the Cauchy regime and their influence on the indices (Section 3.2).
- *SDL versus PV<sub>lo</sub>* (Cells H and I): comparison of the two indices in separating crashes and declines from ordinary fluctuations (Section 3.3, Discussion).

**Notation**
- PV<sub>hi</sub>: proportional variability relative to the higher value of each pair (Heath 2006).
- pv<sub>lo</sub>: unstandardised proportional variability relative to the lower value of each pair.
- PV<sub>lo</sub> = 1 − 1 / (1 + pv<sub>lo</sub>) = pv<sub>lo</sub> / (1 + pv<sub>lo</sub>).
- Zero counts: in a pair containing a zero, the lower value is treated as 1 (one individual). Such a pair contributes 1 to PV<sub>hi</sub> and the positive value itself to pv<sub>lo</sub>; a pair of two zeros contributes 0 to both.

**Usage:** Kernel → *Restart & Run All* (roughly 15–20 minutes on one CPU core). All imports are in the first code cell; if it is skipped, `NameError`s appear. CSV files are written to the `output` folder next to the notebook.


In [1]:
import itertools
import time
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats

# ------------------------------------------------------------------
# Settings
# ------------------------------------------------------------------
N_RUNS      = 50_000         # simulations per condition
N_STEPS     = 100            # time steps per time series
SEED        = 12345          # for reproducibility
OUT_DIR     = Path("output") # CSV files are written here (relative to the notebook folder)
SAVE_SERIES = False          # True: also write the 100 sorted raw values of each run to the CSV (large files!)
DDOF        = 1              # delta degrees of freedom for the SD in CV and SDL:
                             # 1 = sample SD (as in Table 2), 0 = population SD

# Rare condition after Heath (2006): stable N(100, 5), crash N(10, 5)
RARE_FREQ        = 0.05      # crash frequency
RARE_FIXED_COUNT = True      # True: exactly round(RARE_FREQ * N_STEPS) crashes per series;
                             # False: each time step crashes independently with probability RARE_FREQ

OUT_DIR.mkdir(exist_ok=True)


In [2]:
# ------------------------------------------------------------------
# Variability measures
# ------------------------------------------------------------------
def pv_indices(z):
    """Return pv_lo, PV_lo and PV_hi of a time series z (abundances >= 0).

    Efficient method (sorting and cumulative sums, O(n log n)) instead of
    n(n-1)/2 pairwise comparisons. Pairs containing a zero are treated as if
    the lower value were 1 (one individual):
      d_hi(0, z>0) = 1                  -> contribution N0 * Npos
      d_lo(0, z>0) = |z - 0| / 1 = z    -> contribution N0 * sum(z>0)
    Pairs of two zeros contribute 0 to both measures.
    """
    z = np.sort(np.asarray(z, dtype=float))
    n = z.size
    C = n * (n - 1) / 2                         # number of unique pairs
    n0 = int(np.sum(z == 0))                    # number of zeros
    pos = z[z > 0]                              # strictly positive abundances (sorted)
    m = pos.size

    # Comparisons among the positive values
    if m >= 2:
        suffix = np.cumsum(pos[::-1])[::-1]     # suffix[i] = sum of pos[i], ..., pos[m-1]
        sum_pd_lo = np.sum(suffix[1:] / pos[:-1]) - m * (m - 1) / 2
        prefix = np.cumsum(pos)                 # prefix[i] = sum of pos[0], ..., pos[i]
        sum_pd_hi = m * (m - 1) / 2 - np.sum(prefix[:-1] / pos[1:])
    else:
        sum_pd_lo = sum_pd_hi = 0.0

    # Comparisons between zeros and positive values
    sum_pd_lo += n0 * pos.sum()
    sum_pd_hi += n0 * m

    pv_lo = sum_pd_lo / C
    PV_hi = sum_pd_hi / C                       # already bounded between 0 and 1
    PV_lo = 1 - 1 / (1 + pv_lo)
    return pv_lo, PV_lo, PV_hi


def cv_sdl_adm(z):
    """Return CV, SDL and ADM (average deviation from the mean) of a time series z."""
    z = np.asarray(z, dtype=float)
    mean = z.mean()
    cv = z.std(ddof=DDOF) / mean
    # log(0) is undefined: 1 is added to all abundances only if the series contains zeros
    zl = np.log(z + 1) if np.any(z == 0) else np.log(z)
    sdl = zl.std(ddof=DDOF)
    adm = np.mean(np.abs(z - mean))
    return cv, sdl, adm


def all_measures(z):
    cv, sdl, adm = cv_sdl_adm(z)
    pv_lo, PV_lo, PV_hi = pv_indices(z)
    return dict(CV=cv, SDL=sdl, pv_lo=pv_lo, PV_lo=PV_lo, PV_hi=PV_hi, ADM=adm)


In [3]:
# ------------------------------------------------------------------
# Self-tests
# ------------------------------------------------------------------
def pv_bruteforce(z):
    """Pairwise definition from the paper (slow, only for checking): returns pv_lo and PV_hi."""
    z = [float(v) for v in z]
    C = len(z) * (len(z) - 1) / 2
    s_lo = s_hi = 0.0
    for a, b in itertools.combinations(z, 2):
        if a == b:
            continue                            # d_lo = d_hi = 0
        s_hi += abs(a - b) / max(a, b)
        if a > 0 and b > 0:
            s_lo += abs(a - b) / min(a, b)
        else:
            s_lo += abs(a - b)                  # pair with a zero: lower value treated as 1
    return s_lo / C, s_hi / C


# 1) Is the efficient method identical to the pairwise definition (also with zeros)?
_rng = np.random.default_rng(0)
_maxdiff = 0.0
for _ in range(200):
    _z = _rng.integers(0, 30, size=int(_rng.integers(5, 40)))
    _fast = pv_indices(_z)
    _slow = pv_bruteforce(_z)
    _maxdiff = max(_maxdiff, abs(_fast[0] - _slow[0]), abs(_fast[2] - _slow[1]))
print(f"max. deviation efficient vs. pairwise (200 random series, with zeros): {_maxdiff:.2e}")
assert _maxdiff < 1e-9

# 2) Reproduce Table 2 of the paper: CV, SDL, pv_lo, PV_lo, PV_hi
#    Reference values as printed in the paper (rounded to 2 decimals).
TABLE2 = {
    "y1":  ((1, 7, 8, 9, 10, 11, 12, 13, 14, 15),  (0.41, 0.79, 2.32,  0.70, 0.39)),
    "y2":  ((5, 5, 6, 6, 7, 7, 8, 8, 8, 40),       (1.06, 0.60, 1.25,  0.56, 0.32)),
    "y3":  ((5, 5, 8, 8, 10, 10, 12, 12, 15, 15),  (0.36, 0.40, 0.67,  0.40, 0.34)),
    "y4":  ((0, 7, 8, 9, 10, 11, 12, 13, 14, 15),  (0.44, 0.81, 2.52,  0.72, 0.41)),
    "y5":  ((0, 0, 1, 1, 1, 1, 2, 2, 2, 2),        (0.66, 0.42, 0.89,  0.47, 0.53)),
    "y6":  ((0, 0, 1, 3, 10, 12, 15, 17, 20, 22),  (0.85, 1.26, 7.23,  0.88, 0.70)),
    "y7":  ((0, 0, 0, 0, 0, 10, 10, 20, 30, 30),   (1.25, 1.59, 11.36, 0.92, 0.65)),
    "y8":  ((0, 0, 0, 0, 0, 1, 1, 2, 3, 3),        (1.25, 0.60, 1.36,  0.58, 0.65)),
    "y9":  ((0, 0, 0, 0, 0, 0, 0, 0, 1, 99),       (3.13, 1.45, 19.96, 0.95, 0.38)),
    "y10": ((0, 0, 0, 0, 0, 0, 0, 0, 1, 2),        (2.25, 0.39, 0.56,  0.36, 0.37)),
}
TABLE2_COLS = ["CV", "SDL", "pv_lo", "PV_lo", "PV_hi"]

_rows, _problems = [], []
for name, (series, ref) in TABLE2.items():
    cv, sdl, _ = cv_sdl_adm(series)
    pv_lo, PV_lo, PV_hi = pv_indices(series)
    calc = np.array([cv, sdl, pv_lo, PV_lo, PV_hi])
    # reference values are rounded to 2 decimals -> tolerance of half a unit in the last digit
    close = np.abs(calc - np.array(ref)) <= 0.005 + 1e-9
    if DDOF != 1:
        close[:2] = True                        # Table 2 uses the sample SD; CV and SDL not comparable
    _rows.append([name, *np.round(calc, 2), bool(close.all())])
    _problems += [f"{name}: {c} = {v:.4f} (paper: {r})"
                  for c, v, r, ok in zip(TABLE2_COLS, calc, ref, close) if not ok]

table2 = pd.DataFrame(_rows, columns=["series"] + TABLE2_COLS + ["matches paper"])
display(table2)
if DDOF != 1:
    print("Note: DDOF != 1, the CV and SDL columns were not compared with the paper.")
assert not _problems, "Table 2 differs from the paper:\n" + "\n".join(_problems)

# LaTeX rows for Table 2 (to copy into the paper)
print("\nLaTeX rows for Table 2:")
for name, (series, _) in TABLE2.items():
    row = table2.loc[table2.series == name, TABLE2_COLS].iloc[0]
    label = name[0] + "_" + (name[1:] if len(name) == 2 else "{" + name[1:] + "}")
    vec = ", ".join(str(v) for v in series)
    vals = " & ".join(f"{v:.2f}" for v in row)
    print(f"${label} = ({vec})$ & {vals} \\\\")


max. deviation efficient vs. pairwise (200 random series, with zeros): 7.11e-15


,series,CV,SDL,pv_lo,PV_lo,PV_hi,matches paper
0,y1,0.41,0.79,2.32,0.70,0.39,True
1,y2,1.06,0.60,1.25,0.56,0.32,True
2,y3,0.36,0.40,0.67,0.40,0.34,True
3,y4,0.44,0.81,2.52,0.72,0.41,True
4,y5,0.66,0.42,0.89,0.47,0.53,True
5,y6,0.85,1.26,7.23,0.88,0.70,True
6,y7,1.25,1.59,11.36,0.92,0.65,True
7,y8,1.25,0.60,1.36,0.58,0.65,True
8,y9,3.13,1.45,19.96,0.95,0.38,True
9,y10,2.25,0.39,0.56,0.36,0.37,True



LaTeX rows for Table 2:
$y_1 = (1, 7, 8, 9, 10, 11, 12, 13, 14, 15)$ & 0.41 & 0.79 & 2.32 & 0.70 & 0.39 \\
$y_2 = (5, 5, 6, 6, 7, 7, 8, 8, 8, 40)$ & 1.06 & 0.60 & 1.25 & 0.56 & 0.32 \\
$y_3 = (5, 5, 8, 8, 10, 10, 12, 12, 15, 15)$ & 0.36 & 0.40 & 0.67 & 0.40 & 0.34 \\
$y_4 = (0, 7, 8, 9, 10, 11, 12, 13, 14, 15)$ & 0.44 & 0.81 & 2.52 & 0.72 & 0.41 \\
$y_5 = (0, 0, 1, 1, 1, 1, 2, 2, 2, 2)$ & 0.66 & 0.42 & 0.89 & 0.47 & 0.53 \\
$y_6 = (0, 0, 1, 3, 10, 12, 15, 17, 20, 22)$ & 0.85 & 1.26 & 7.23 & 0.88 & 0.70 \\
$y_7 = (0, 0, 0, 0, 0, 10, 10, 20, 30, 30)$ & 1.25 & 1.59 & 11.36 & 0.92 & 0.65 \\
$y_8 = (0, 0, 0, 0, 0, 1, 1, 2, 3, 3)$ & 1.25 & 0.60 & 1.36 & 0.58 & 0.65 \\
$y_9 = (0, 0, 0, 0, 0, 0, 0, 0, 1, 99)$ & 3.13 & 1.45 & 19.96 & 0.95 & 0.38 \\
$y_{10} = (0, 0, 0, 0, 0, 0, 0, 0, 1, 2)$ & 2.25 & 0.39 & 0.56 & 0.36 & 0.37 \\


In [4]:
# ------------------------------------------------------------------
# Generators for the simulated time series
# ------------------------------------------------------------------
def draw_normal_pos(rng, mean, sd, size):
    """Rounded normal distribution; values < 1 are redrawn (abundances = natural numbers)."""
    x = np.round(rng.normal(mean, sd, size))
    bad = x < 1
    while bad.any():
        x[bad] = np.round(rng.normal(mean, sd, bad.sum()))
        bad = x < 1
    return x


def gen_normal(rng):
    return draw_normal_pos(rng, 1000, 150, N_STEPS)


def gen_cauchy(rng):
    # Cauchy (location 100, scale 5), rounded; "natural numbers": absolute value, lower bound 1
    x = np.round(100 + 5 * rng.standard_cauchy(N_STEPS))
    return np.maximum(np.abs(x), 1)


def gen_rare(rng):
    # stable N(100, 5); crash N(10, 5) with frequency RARE_FREQ (Heath 2006)
    if RARE_FIXED_COUNT:
        n_crash = int(round(RARE_FREQ * N_STEPS))
        crash = np.zeros(N_STEPS, dtype=bool)
        crash[rng.choice(N_STEPS, n_crash, replace=False)] = True
    else:
        crash = rng.random(N_STEPS) < RARE_FREQ
    x = np.empty(N_STEPS)
    x[~crash] = draw_normal_pos(rng, 100, 5, (~crash).sum())
    x[crash] = draw_normal_pos(rng, 10, 5, crash.sum())
    return x


def gen_bimodal(rng):
    # first half N(100, 10), second half N(10, 1)
    half = N_STEPS // 2
    return np.concatenate([draw_normal_pos(rng, 100, 10, half),
                           draw_normal_pos(rng, 10, 1, N_STEPS - half)])


CONDITIONS = {"Normal": gen_normal, "Cauchy": gen_cauchy, "Rare": gen_rare, "Bimodal": gen_bimodal}


In [5]:
# ------------------------------------------------------------------
# Simulation
# ------------------------------------------------------------------
rng = np.random.default_rng(SEED)
runs = {}

for cond, gen in CONDITIONS.items():
    t0 = time.time()
    records, series = [], []
    for _ in range(N_RUNS):
        z = gen(rng)
        records.append(all_measures(z))
        if SAVE_SERIES:
            series.append(np.sort(z))
    df = pd.DataFrame.from_records(records)
    if SAVE_SERIES:
        df = pd.concat([df, pd.DataFrame(series, columns=[f"sortval{i}" for i in range(1, N_STEPS + 1)])], axis=1)
    runs[cond] = df
    df.to_csv(OUT_DIR / f"runs_{cond}.csv", index=False)
    print(f"{cond:8s}: {N_RUNS} runs in {time.time() - t0:5.1f} s  ->  {OUT_DIR / f'runs_{cond}.csv'}")


Normal  : 50000 runs in   5.4 s  ->  output\runs_Normal.csv
Cauchy  : 50000 runs in   6.5 s  ->  output\runs_Cauchy.csv
Rare    : 50000 runs in   9.5 s  ->  output\runs_Rare.csv
Bimodal : 50000 runs in   8.2 s  ->  output\runs_Bimodal.csv


In [6]:
# ------------------------------------------------------------------
# Summary: mean, SD and correlation with ADM (Pearson, Spearman, Kendall)
# ------------------------------------------------------------------
measures = ["CV", "SDL", "pv_lo", "PV_lo", "PV_hi"]
rows = []
for cond, df in runs.items():
    for m in measures:
        r, _   = stats.pearsonr(df[m], df["ADM"])
        rho, _ = stats.spearmanr(df[m], df["ADM"])
        tau, _ = stats.kendalltau(df[m], df["ADM"])
        rows.append(dict(condition=cond, measure=m, mean=df[m].mean(), sd=df[m].std(),
                         pearson_ADM=r, spearman_ADM=rho, kendall_ADM=tau))
summary = pd.DataFrame(rows)
summary.to_csv(OUT_DIR / "summary.csv", index=False)
display(summary.round(3))

# Compact views: means and SDs (bar chart of the variability indices), correlations with ADM
display(summary.pivot(index="measure", columns="condition", values="mean").loc[measures, list(CONDITIONS)].round(3))
display(summary.pivot(index="measure", columns="condition", values="sd").loc[measures, list(CONDITIONS)].round(3))
display(summary.pivot(index="measure", columns="condition", values="pearson_ADM").loc[measures, list(CONDITIONS)].round(3))
display(summary.pivot(index="measure", columns="condition", values="spearman_ADM").loc[measures, list(CONDITIONS)].round(3))


,condition,measure,mean,sd,pearson_ADM,spearman_ADM,kendall_ADM
0,Normal,CV,0.150,0.011,0.917,0.910,0.740
1,Normal,SDL,0.154,0.012,0.863,0.858,0.672
2,Normal,pv_lo,0.200,0.018,0.917,0.913,0.745
3,Normal,PV_lo,0.166,0.012,0.918,0.913,0.745
4,Normal,PV_hi,0.151,0.010,0.948,0.943,0.795
5,Cauchy,CV,0.951,1.351,0.117,0.976,0.868
6,Cauchy,SDL,0.366,0.132,0.090,0.707,0.537
7,Cauchy,pv_lo,1.344,30.939,0.998,0.696,0.540
8,Cauchy,PV_lo,0.379,0.177,0.066,0.696,0.540
9,Cauchy,PV_hi,0.171,0.027,0.016,0.602,0.437


condition,Normal,Cauchy,Rare,Bimodal
measure,,,,
CV,0.150,0.951,0.212,0.832
SDL,0.154,0.366,0.540,1.162
pv_lo,0.200,1.344,1.287,4.664
PV_lo,0.166,0.379,0.540,0.823
PV_hi,0.151,0.171,0.136,0.505


condition,Normal,Cauchy,Rare,Bimodal
measure,,,,
CV,0.011,1.351,0.005,0.004
SDL,0.012,0.132,0.063,0.010
pv_lo,0.018,30.939,0.593,0.107
PV_lo,0.012,0.177,0.091,0.003
PV_hi,0.010,0.027,0.004,0.004


condition,Normal,Cauchy,Rare,Bimodal
measure,,,,
CV,0.917,0.117,0.702,0.612
SDL,0.863,0.090,0.518,0.752
pv_lo,0.917,0.998,0.398,0.742
PV_lo,0.918,0.066,0.464,0.742
PV_hi,0.948,0.016,0.878,0.142


condition,Normal,Cauchy,Rare,Bimodal
measure,,,,
CV,0.910,0.976,0.686,0.593
SDL,0.858,0.707,0.516,0.734
pv_lo,0.913,0.696,0.482,0.723
PV_lo,0.913,0.696,0.482,0.723
PV_hi,0.943,0.602,0.869,0.135


In [7]:
# ------------------------------------------------------------------
# Additional analysis (Rare regime): what drives the indices, and what
# would they be without crashes?
# Requires the cells above (all_measures, draw_normal_pos, N_STEPS, np, pd, stats).
# A separate random number generator (rng_extra) is used, so the main
# results above are not affected.
# ------------------------------------------------------------------
N_EXTRA   = 20_000                       # simulated series (smaller than N_RUNS, sufficient for rank correlations)
N_CRASHES = round(RARE_FREQ * N_STEPS)   # 5 crashes per series, as in gen_rare()
rng_extra = np.random.default_rng(7)

# --- Part A: between-run drivers ----------------------------------------------
# For every simulated Rare series we record two features of the series:
#   sd_stable      : SD of the stable (non-crash) abundances -> scatter of the stable phase
#   mean_inv_crash : mean of 1/z over the crashed abundances -> proportional depth of the
#                    crashes (low values receive a large weight, as in pv_lo, Eq. 10 of the paper)
rows = []
for _ in range(N_EXTRA):
    crash = np.zeros(N_STEPS, dtype=bool)
    crash[rng_extra.choice(N_STEPS, N_CRASHES, replace=False)] = True
    z = np.empty(N_STEPS)
    z[~crash] = draw_normal_pos(rng_extra, 100, 5, (~crash).sum())    # stable phase  N(100, 5)
    z[crash]  = draw_normal_pos(rng_extra, 10, 5, crash.sum())        # crashes       N(10, 5)
    row = all_measures(z)
    row["mean_inv_crash"] = np.mean(1 / z[crash])
    row["sd_stable"]      = z[~crash].std(ddof=1)
    rows.append(row)
rare_extra = pd.DataFrame(rows)

# Spearman correlation of each index (and of ADM) with the two features, across the series.
# Spearman uses ranks only, so pv_lo and PV_lo give identical values.
drivers = pd.DataFrame({
    m: {f: stats.spearmanr(rare_extra[m], rare_extra[f])[0] for f in ["sd_stable", "mean_inv_crash"]}
    for m in ["CV", "SDL", "PV_lo", "PV_hi", "ADM"]
}).T
print("Spearman correlation of each index with features of the Rare series:")
display(drivers.round(2))

# --- Part B: reference without crashes ----------------------------------------
# Same stable population N(100, 5) as in the Rare regime, but without any crash.
# This is NOT the Normal regime of the paper (mean 1000, SD 150): the relative fluctuation
# is 5 % here (CV about 0.05) instead of 15 %, so the index values are lower.
# It shows how much of the Rare-regime scores is caused by the crashes themselves.
no_crash = pd.DataFrame([all_measures(draw_normal_pos(rng_extra, 100, 5, N_STEPS))
                         for _ in range(N_EXTRA)])
reference = pd.DataFrame({
    "no crash": no_crash[["CV", "SDL", "PV_lo", "PV_hi"]].mean(),
    "Rare (5 crashes)": runs["Rare"][["CV", "SDL", "PV_lo", "PV_hi"]].mean(),
})
reference["factor"] = reference["Rare (5 crashes)"] / reference["no crash"]
print("\nMean index values without crashes and in the Rare regime:")
display(reference.round(3))


Spearman correlation of each index with features of the Rare series:


,sd_stable,mean_inv_crash
CV,0.18,0.82
SDL,0.01,0.99
PV_lo,0.02,1.00
PV_hi,0.84,0.43
ADM,0.66,0.47



Mean index values without crashes and in the Rare regime:


,no crash,Rare (5 crashes),factor
CV,0.050,0.212,4.242
SDL,0.050,0.540,10.776
PV_lo,0.056,0.540,9.669
PV_hi,0.054,0.136,2.504


In [8]:
# ------------------------------------------------------------------
# OPTIONAL extra cell: dependence of the indices on the length of the time series
# ("more time - more variation"; Heath 2006, Inchausti & Halley 2002).
# Requires the cells above (all_measures, CONDITIONS, N_STEPS, np, pd).
# A separate random number generator is used, so the main results are not affected.
# ------------------------------------------------------------------
# Idea: a good index should not increase just because a series is observed for longer.
# For every simulated series we compute each index in windows of length k (k = 3 ... 100),
# average it over several random windows per k, and fit a straight line to
# ln(index) against ln(k). The slope is the variance exponent gamma:
#   gamma = 0  : index does not depend on series length (white-noise-like behaviour)
#   gamma > 0  : index increases with series length ("more time - more variation")
# Only the stationary regimes are used (Bimodal is non-stationary by construction).
N_SERIES   = 1000                                   # series per regime (as in Heath 2006)
N_WINDOWS  = 15                                     # random windows per window length k
KS         = [3, 4, 6, 9, 13, 20, 30, 45, 70, 100]  # window lengths (time steps)
MEASURES   = ["CV", "SDL", "PV_lo", "PV_hi"]
rng_exp    = np.random.default_rng(2024)

def variance_exponents(gen, n_series=N_SERIES):
    gammas = {m: [] for m in MEASURES}
    for _ in range(n_series):
        z = gen(rng_exp)                            # one simulated series of N_STEPS values
        mean_index = {m: [] for m in MEASURES}      # index averaged over windows, per k
        for k in KS:
            starts = rng_exp.integers(0, N_STEPS - k + 1, size=N_WINDOWS) if k < N_STEPS else [0]
            vals = [all_measures(z[s:s + k]) for s in starts]
            for m in MEASURES:
                mean_index[m].append(np.mean([v[m] for v in vals]))
        for m in MEASURES:                          # slope of ln(index) vs ln(k)
            gammas[m].append(np.polyfit(np.log(KS), np.log(mean_index[m]), 1)[0])
    return pd.DataFrame(gammas)

rows = {}
for cond in ["Normal", "Cauchy", "Rare"]:
    g = variance_exponents(CONDITIONS[cond])
    rows[cond] = {f"{m} mean": g[m].mean() for m in MEASURES} | {f"{m} SD": g[m].std() for m in MEASURES}
print("Variance exponents gamma (mean and SD across simulated series):")
display(pd.DataFrame(rows).T.round(3))

Variance exponents gamma (mean and SD across simulated series):


,CV mean,SDL mean,PV_lo mean,PV_hi mean,CV SD,SDL SD,PV_lo SD,PV_hi SD
Normal,0.030,0.035,0.015,0.003,0.027,0.029,0.026,0.025
Cauchy,0.334,0.174,0.185,0.008,0.254,0.093,0.124,0.046
Rare,0.145,0.287,0.359,0.022,0.093,0.121,0.103,0.077


## Decline regime: parameter exploration (Cells A and B)

Gradual-decline regime: the population fluctuates around 100 and, after a fraction `onset` of the series, declines gradually to `100 / factor` at the last time step (`"exp"`: constant per-capita rate, `"lin"`: linear). Scatter around the path is Gaussian with relative SD `noise` (values < 1 are redrawn, as elsewhere in this notebook). Cells A and B explore combinations of `factor`, `onset` and `noise`. They served to choose the parameters of Section 3.3 (main case: factor 10, onset 0.5, noise 0.15; stress test: noise 0.30; sensitivity: factor 100) and are not reported in the paper. The final analysis is in Cell C.

- **Cell A** (grid): index means, response relative to a stable reference series with the same noise, and rank discrimination (AUC) between decline depths.
- **Cell B** (expanding window): the series is revealed step by step; an index "detects" the decline when it exceeds the 95th percentile of the same index in stable reference series of the same length (one-sided, no multiplicity correction; exploratory).

The consecutive disparity index D (Fernández-Martínez et al. 2018) is included as an order-sensitive reference (ln-ratios of consecutive values; 1 is added to all values only if the series contains zeros).

In [9]:
# ------------------------------------------------------------------
# Cell A: decline regime, grid over final density (factor), onset and noise
# Requires the cells above (all_measures, draw_normal_pos, N_STEPS, np, pd, stats, OUT_DIR).
# ------------------------------------------------------------------
DECL_FACTORS = [2, 5, 10, 20, 50, 100]   # final abundance = 100 / factor
DECL_ONSETS  = [0.3, 0.5, 0.7]           # decline starts after this fraction of the series
DECL_NOISES  = [0.05, 0.15, 0.30]        # relative SD of the scatter around the path (0.05: Rare stable phase, 0.15: Normal)
DECL_SHAPE   = "exp"                     # "exp": constant per-capita decline rate; "lin": linear decline
N_DECL       = 5000                      # simulated series per combination
rng_dec      = np.random.default_rng(4711)   # separate generator: main results are not affected

def d_index(z):
    """Consecutive disparity index D: mean |ln(z[t+1] / z[t])| (1 is added if the series contains zeros)."""
    z = np.asarray(z, dtype=float)
    if np.any(z == 0):
        z = z + 1
    return np.mean(np.abs(np.diff(np.log(z))))

def draw_around_path(rng, mean, rel_sd):
    """Rounded Gaussian scatter around a deterministic path; values < 1 are redrawn."""
    x = np.round(rng.normal(mean, rel_sd * mean))
    bad = x < 1
    while bad.any():
        x[bad] = np.round(rng.normal(mean[bad], rel_sd * mean[bad]))
        bad = x < 1
    return x

def gen_decline(rng, factor, onset, noise, n=N_STEPS, shape=DECL_SHAPE):
    t0 = int(round(onset * n))
    path = np.full(n, 100.0)
    m = n - t0
    if m > 0:
        s = np.arange(1, m + 1) / m             # 0 -> 1 over the decline phase
        path[t0:] = 100.0 * factor ** (-s) if shape == "exp" else 100.0 - (100.0 - 100.0 / factor) * s
    return draw_around_path(rng, path, noise)

def measures_with_d(z):
    r = all_measures(z)
    r["D"] = d_index(z)
    return r

DECL_MEASURES = ["CV", "SDL", "pv_lo", "PV_lo", "PV_hi", "D"]
RESP_MEASURES = ["CV", "SDL", "PV_lo", "PV_hi", "D"]

decl_runs, decl_ref, rows = {}, {}, []
for noise in DECL_NOISES:
    ref = pd.DataFrame([measures_with_d(draw_normal_pos(rng_dec, 100, 100 * noise, N_STEPS))
                        for _ in range(N_DECL)])
    decl_ref[noise] = ref
    for onset in DECL_ONSETS:
        for F in DECL_FACTORS:
            df = pd.DataFrame([measures_with_d(gen_decline(rng_dec, F, onset, noise)) for _ in range(N_DECL)])
            decl_runs[(noise, onset, F)] = df
            rows.append(dict(noise=noise, onset=onset, factor=F, **df[DECL_MEASURES].mean().to_dict()))
decl_grid = pd.DataFrame(rows).set_index(["noise", "onset", "factor"])
decl_grid.to_csv(OUT_DIR / "decline_grid.csv")

print("Mean index values (end of the series, n = %d):" % N_STEPS)
display(decl_grid.round(3))

# response relative to a stable series with the same noise (same logic as 'factor' in the Rare analysis)
resp = decl_grid.copy()
for noise in DECL_NOISES:
    ref_mean = decl_ref[noise][DECL_MEASURES].mean()
    resp.loc[noise] = decl_grid.loc[noise].values / ref_mean.values
resp = resp[RESP_MEASURES]
resp["PV_lo / PV_hi"] = resp["PV_lo"] / resp["PV_hi"]
print("\nResponse: index mean divided by the mean of a stable reference series with the same noise:")
display(resp.round(2))

# depth resolution: P(index of the deeper decline > index of the shallower decline) (AUC), rank-based
def auc(a, b):
    return stats.mannwhitneyu(b, a).statistic / (len(a) * len(b))

PAIRS = [(2, 10), (10, 100), (20, 50), (50, 100)]
arows = []
for noise in DECL_NOISES:
    for onset in DECL_ONSETS:
        for f1, f2 in PAIRS:
            r = dict(noise=noise, onset=onset, pair=f"{f1} vs {f2}")
            for m in RESP_MEASURES:
                r[m] = auc(decl_runs[(noise, onset, f1)][m], decl_runs[(noise, onset, f2)][m])
            arows.append(r)
        r = dict(noise=noise, onset=onset, pair="stable vs 2")   # detectability of a shallow decline
        for m in RESP_MEASURES:
            r[m] = auc(decl_ref[noise][m], decl_runs[(noise, onset, 2)][m])
        arows.append(r)
decl_auc = pd.DataFrame(arows).set_index(["noise", "onset", "pair"])
decl_auc.to_csv(OUT_DIR / "decline_auc.csv")
print("\nAUC (1 = deeper decline always ranks higher, 0.5 = no discrimination):")
display(decl_auc.round(3))

Mean index values (end of the series, n = 100):


CV    SDL   pv_lo  PV_lo  PV_hi      D
noise onset factor                                           
0.05  0.3   2       0.226  0.239   0.341  0.254  0.224  0.057
            5       0.470  0.545   1.073  0.518  0.406  0.059
            10      0.619  0.777   2.031  0.670  0.499  0.061
            20      0.740  1.011   3.618  0.783  0.566  0.064
            50      0.865  1.324   7.614  0.884  0.631  0.068
            100     0.940  1.568  13.706  0.932  0.667  0.073
      0.5   2       0.210  0.233   0.319  0.242  0.208  0.057
            5       0.409  0.531   1.004  0.501  0.366  0.059
            10      0.518  0.758   1.923  0.658  0.444  0.062
            20      0.600  0.985   3.476  0.777  0.500  0.066
            50      0.679  1.289   7.459  0.882  0.552  0.072
            100     0.723  1.526  13.636  0.932  0.581  0.078
      0.7   2       0.175  0.204   0.252  0.201  0.167  0.057
            5       0.320  0.463   0.768  0.434  0.279  0.061
            10      0.393  0.661   1.472  0.595  0.333  0.066
            20      0.444  0.859   2.677  0.728  0.372  0.071
            50      0.492  1.123   5.810  0.853  0.407  0.080
            100     0.517  1.328  10.707  0.915  0.426  0.086
0.15  0.3   2       0.268  0.280   0.417  0.294  0.255  0.173
            5       0.495  0.564   1.145  0.534  0.421  0.173
            10      0.640  0.791   2.120  0.679  0.510  0.174
            20      0.760  1.021   3.738  0.789  0.576  0.175
            50      0.885  1.329   7.759  0.886  0.639  0.177
            100     0.959  1.572  13.861  0.933  0.675  0.178
      0.5   2       0.255  0.275   0.404  0.288  0.247  0.173
            5       0.436  0.550   1.088  0.521  0.393  0.174
            10      0.541  0.772   2.023  0.669  0.469  0.175
            20      0.622  0.996   3.606  0.783  0.524  0.176
            50      0.699  1.295   7.625  0.884  0.576  0.178
            100     0.743  1.530  13.803  0.932  0.604  0.180
      0.7   2       0.226  0.251   0.353  0.261  0.223  0.173
            5       0.353  0.486   0.871  0.465  0.328  0.174
            10      0.421  0.677   1.588  0.613  0.381  0.176
            20      0.470  0.871   2.817  0.738  0.419  0.178
            50      0.516  1.131   5.992  0.857  0.453  0.182
            100     0.540  1.334  10.903  0.916  0.473  0.185
0.30  0.3   2       0.376  0.420   0.750  0.423  0.331  0.371
            5       0.571  0.645   1.494  0.596  0.455  0.372
            10      0.707  0.850   2.543  0.716  0.533  0.372
            20      0.824  1.067   4.306  0.810  0.594  0.372
            50      0.947  1.363   8.661  0.896  0.654  0.372
            100     1.021  1.586  14.439  0.935  0.687  0.364
      0.5   2       0.367  0.418   0.748  0.422  0.328  0.373
            5       0.517  0.634   1.453  0.589  0.440  0.373
            10      0.614  0.833   2.462  0.709  0.508  0.373
            20      0.690  1.044   4.199  0.806  0.559  0.373
            50      0.765  1.330   8.529  0.894  0.608  0.374
            100     0.808  1.547  14.391  0.935  0.636  0.368
      0.7   2       0.346  0.402   0.703  0.407  0.316  0.372
            5       0.444  0.578   1.238  0.549  0.397  0.372
            10      0.504  0.746   2.020  0.666  0.446  0.373
            20      0.548  0.925   3.363  0.769  0.481  0.374
            50      0.590  1.173   6.809  0.871  0.515  0.376
            100     0.613  1.359  11.457  0.919  0.534  0.373


Response: index mean divided by the mean of a stable reference series with the same noise:


CV    SDL  PV_lo  PV_hi     D  PV_lo / PV_hi
noise onset factor                                                 
0.05  0.3   2        4.53   4.77   4.55   4.14  1.01           1.10
            5        9.40  10.87   9.27   7.52  1.04           1.23
            10      12.39  15.52  12.00   9.22  1.08           1.30
            20      14.82  20.18  14.03  10.47  1.12           1.34
            50      17.33  26.42  15.83  11.66  1.21           1.36
            100     18.81  31.30  16.70  12.34  1.29           1.35
      0.5   2        4.21   4.65   4.33   3.84  1.01           1.13
            5        8.18  10.59   8.98   6.77  1.05           1.33
            10      10.36  15.12  11.78   8.21  1.11           1.43
            20      12.01  19.67  13.91   9.25  1.17           1.50
            50      13.59  25.74  15.80  10.21  1.28           1.55
            100     14.47  30.47  16.69  10.75  1.38           1.55
      0.7   2        3.51   4.08   3.61   3.08  1.02           1.17
            5        6.41   9.24   7.78   5.16  1.08           1.51
            10       7.86  13.19  10.67   6.17  1.16           1.73
            20       8.90  17.14  13.04   6.87  1.26           1.90
            50       9.84  22.42  15.28   7.52  1.41           2.03
            100     10.35  26.52  16.38   7.88  1.53           2.08
0.15  0.3   2        1.79   1.82   1.77   1.68  1.00           1.05
            5        3.31   3.66   3.21   2.78  1.00           1.15
            10       4.28   5.14   4.09   3.37  1.01           1.21
            20       5.08   6.63   4.75   3.81  1.02           1.25
            50       5.92   8.63   5.33   4.23  1.02           1.26
            100      6.41  10.21   5.61   4.47  1.03           1.26
      0.5   2        1.70   1.78   1.73   1.63  1.00           1.06
            5        2.91   3.57   3.13   2.60  1.01           1.20
            10       3.62   5.01   4.03   3.10  1.01           1.30
            20       4.16   6.47   4.71   3.47  1.02           1.36
            50       4.68   8.41   5.32   3.81  1.03           1.40
            100      4.97   9.94   5.61   4.00  1.04           1.40
      0.7   2        1.51   1.63   1.57   1.47  1.00           1.07
            5        2.36   3.15   2.80   2.17  1.01           1.29
            10       2.81   4.40   3.69   2.52  1.02           1.46
            20       3.14   5.66   4.44   2.77  1.03           1.60
            50       3.45   7.35   5.15   3.00  1.05           1.72
            100      3.61   8.67   5.51   3.12  1.07           1.76
0.30  0.3   2        1.26   1.20   1.18   1.19  1.00           1.00
            5        1.91   1.85   1.67   1.63  1.00           1.02
            10       2.37   2.43   2.00   1.91  1.00           1.05
            20       2.76   3.05   2.27   2.13  1.00           1.06
            50       3.17   3.90   2.51   2.35  1.00           1.07
            100      3.42   4.54   2.62   2.46  0.98           1.06
      0.5   2        1.23   1.20   1.18   1.18  1.00           1.00
            5        1.73   1.82   1.65   1.58  1.00           1.04
            10       2.06   2.38   1.98   1.82  1.00           1.09
            20       2.31   2.99   2.26   2.01  1.00           1.12
            50       2.56   3.81   2.50   2.18  1.00           1.15
            100      2.71   4.43   2.62   2.28  0.99           1.15
      0.7   2        1.16   1.15   1.14   1.13  1.00           1.00
            5        1.49   1.65   1.54   1.43  1.00           1.08
            10       1.69   2.13   1.86   1.60  1.00           1.16
            20       1.84   2.65   2.15   1.73  1.01           1.24
            50       1.98   3.36   2.44   1.85  1.01           1.32
            100      2.05   3.89   2.57   1.91  1.00           1.34


AUC (1 = deeper decline always ranks higher, 0.5 = no discrimination):


CV    SDL  PV_lo  PV_hi      D
noise onset pair                                          
0.05  0.3   2 vs 10      1.000  1.000  1.000  1.000  0.699
            10 vs 100    1.000  1.000  1.000  1.000  0.932
            20 vs 50     1.000  1.000  1.000  1.000  0.713
            50 vs 100    1.000  1.000  1.000  1.000  0.682
            stable vs 2  1.000  1.000  1.000  1.000  0.524
      0.5   2 vs 10      1.000  1.000  1.000  1.000  0.764
            10 vs 100    1.000  1.000  1.000  1.000  0.986
            20 vs 50     1.000  1.000  1.000  1.000  0.807
            50 vs 100    1.000  1.000  1.000  1.000  0.774
            stable vs 2  1.000  1.000  1.000  1.000  0.537
      0.7   2 vs 10      1.000  1.000  1.000  1.000  0.878
            10 vs 100    1.000  1.000  1.000  1.000  0.999
            20 vs 50     1.000  1.000  1.000  1.000  0.902
            50 vs 100    1.000  1.000  1.000  1.000  0.853
            stable vs 2  1.000  1.000  1.000  1.000  0.553
0.15  0.3   2 vs 10      1.000  1.000  1.000  1.000  0.523
            10 vs 100    1.000  1.000  1.000  1.000  0.558
            20 vs 50     1.000  1.000  1.000  1.000  0.523
            50 vs 100    1.000  1.000  1.000  1.000  0.519
            stable vs 2  1.000  1.000  1.000  1.000  0.505
      0.5   2 vs 10      1.000  1.000  1.000  1.000  0.533
            10 vs 100    1.000  1.000  1.000  1.000  0.579
            20 vs 50     1.000  1.000  1.000  1.000  0.536
            50 vs 100    1.000  1.000  1.000  1.000  0.523
            stable vs 2  1.000  1.000  1.000  1.000  0.508
      0.7   2 vs 10      1.000  1.000  1.000  1.000  0.561
            10 vs 100    1.000  1.000  1.000  1.000  0.652
            20 vs 50     1.000  1.000  1.000  1.000  0.557
            50 vs 100    0.999  1.000  1.000  0.982  0.561
            stable vs 2  1.000  1.000  1.000  1.000  0.499
0.30  0.3   2 vs 10      1.000  1.000  0.998  1.000  0.506
            10 vs 100    1.000  1.000  1.000  1.000  0.445
            20 vs 50     1.000  1.000  1.000  1.000  0.503
            50 vs 100    0.986  1.000  1.000  1.000  0.445
            stable vs 2  0.991  0.875  0.877  0.979  0.496
      0.5   2 vs 10      1.000  1.000  0.996  1.000  0.503
            10 vs 100    1.000  1.000  1.000  1.000  0.466
            20 vs 50     0.998  1.000  1.000  1.000  0.503
            50 vs 100    0.949  1.000  1.000  0.996  0.459
            stable vs 2  0.981  0.866  0.869  0.971  0.503
      0.7   2 vs 10      1.000  1.000  0.993  1.000  0.510
            10 vs 100    1.000  1.000  1.000  1.000  0.505
            20 vs 50     0.955  1.000  1.000  0.981  0.511
            50 vs 100    0.832  1.000  0.999  0.876  0.483
            stable vs 2  0.933  0.810  0.815  0.927  0.495

In [10]:
# ------------------------------------------------------------------
# Cell B: decline regime, expanding window ("monitoring that accumulates over time")
# Requires Cell A (gen_decline, measures_with_d, draw_normal_pos, rng_dec, ...).
# For every prefix length k the index is compared with the 95th percentile of the same index
# in stable reference series of length k (same noise). Power = share of decline series above it.
# ------------------------------------------------------------------
DET_FACTORS = [2, 10, 100]
DET_ONSETS  = [0.3, 0.5, 0.7]
DET_NOISES  = [0.15, 0.30]
DET_DELAYS  = [10, 20, 30]                       # time steps after the onset of the decline
DET_KS      = list(range(10, N_STEPS + 1, 2))    # prefix lengths
DET_MEAS    = ["CV", "SDL", "PV_lo", "PV_hi", "D"]
N_DET_REF, N_DET, DET_Q = 2000, 1000, 0.95

drows = []
for noise in DET_NOISES:
    ref = {m: np.empty((N_DET_REF, len(DET_KS))) for m in DET_MEAS}
    for i in range(N_DET_REF):
        z = draw_normal_pos(rng_dec, 100, 100 * noise, N_STEPS)
        for j, k in enumerate(DET_KS):
            r = measures_with_d(z[:k])
            for m in DET_MEAS:
                ref[m][i, j] = r[m]
    thr = {m: np.quantile(ref[m], DET_Q, axis=0) for m in DET_MEAS}
    for onset in DET_ONSETS:
        t0 = int(round(onset * N_STEPS))
        for F in DET_FACTORS:
            hit = {m: np.zeros((N_DET, len(DET_KS)), dtype=bool) for m in DET_MEAS}
            for i in range(N_DET):
                z = gen_decline(rng_dec, F, onset, noise)
                for j, k in enumerate(DET_KS):
                    r = measures_with_d(z[:k])
                    for m in DET_MEAS:
                        hit[m][i, j] = r[m] > thr[m][j]
            for m in DET_MEAS:
                row = dict(noise=noise, onset=onset, factor=F, measure=m)
                for d in DET_DELAYS:
                    if t0 + d <= N_STEPS:
                        row[f"power +{d}"] = hit[m][:, DET_KS.index(t0 + d)].mean()
                row["power end"] = hit[m][:, -1].mean()
                drows.append(row)
detect = pd.DataFrame(drows)
detect.to_csv(OUT_DIR / "decline_detection.csv", index=False)
print(f"Share of decline series above the {DET_Q:.0%} threshold of stable series of the same length")
print("(+d = d time steps after the onset of the decline; false-positive rate of the threshold is 5 % per prefix length):")
display(detect.pivot_table(index=["noise", "onset", "factor"], columns="measure",
                           values=["power +10", "power +20", "power +30", "power end"])
        [["power +10", "power +20", "power end"]].round(2))

Share of decline series above the 95% threshold of stable series of the same length
(+d = d time steps after the onset of the decline; false-positive rate of the threshold is 5 % per prefix length):


power +10                         power +20              \
measure                   CV     D PV_hi PV_lo   SDL        CV     D PV_hi   
noise onset factor                                                           
0.15  0.3   2           0.07  0.05  0.07  0.06  0.06      0.19  0.05  0.20   
            10          0.36  0.05  0.38  0.35  0.34      1.00  0.05  1.00   
            100         0.98  0.07  0.99  0.99  0.99      1.00  0.05  1.00   
      0.5   2           0.07  0.05  0.07  0.06  0.07      0.33  0.04  0.33   
            10          0.61  0.04  0.61  0.66  0.68      1.00  0.05  1.00   
            100         1.00  0.05  1.00  1.00  1.00      1.00  0.08  1.00   
      0.7   2           0.14  0.06  0.14  0.15  0.14      0.89  0.05  0.90   
            10          0.98  0.04  0.98  1.00  1.00      1.00  0.05  1.00   
            100         1.00  0.05  1.00  1.00  1.00      1.00  0.10  1.00   
0.30  0.3   2           0.06  0.06  0.07  0.06  0.06      0.08  0.06  0.09   
            10          0.10  0.05  0.10  0.06  0.06      0.53  0.06  0.48   
            100         0.29  0.06  0.26  0.12  0.12      1.00  0.06  1.00   
      0.5   2           0.07  0.07  0.08  0.06  0.06      0.12  0.07  0.12   
            10          0.14  0.06  0.14  0.06  0.07      0.84  0.05  0.84   
            100         0.50  0.08  0.53  0.20  0.21      1.00  0.07  1.00   
      0.7   2           0.08  0.05  0.07  0.06  0.06      0.20  0.05  0.19   
            10          0.25  0.04  0.25  0.07  0.08      1.00  0.04  1.00   
            100         0.90  0.05  0.92  0.63  0.72      1.00  0.04  1.00   

                               power end                          
measure            PV_lo   SDL        CV     D PV_hi PV_lo   SDL  
noise onset factor                                                
0.15  0.3   2       0.19  0.18      1.00  0.04  1.00  1.00  1.00  
            10      1.00  1.00      1.00  0.04  1.00  1.00  1.00  
            100     1.00  1.00      1.00  0.10  1.00  1.00  1.00  
      0.5   2       0.33  0.30      1.00  0.04  1.00  1.00  1.00  
            10      1.00  1.00      1.00  0.05  1.00  1.00  1.00  
            100     1.00  1.00      1.00  0.11  1.00  1.00  1.00  
      0.7   2       0.92  0.91      1.00  0.05  1.00  1.00  1.00  
            10      1.00  1.00      1.00  0.06  1.00  1.00  1.00  
            100     1.00  1.00      1.00  0.16  1.00  1.00  1.00  
0.30  0.3   2       0.08  0.08      0.95  0.06  0.91  0.23  0.28  
            10      0.14  0.15      1.00  0.06  1.00  1.00  1.00  
            100     0.95  0.96      1.00  0.03  1.00  1.00  1.00  
      0.5   2       0.06  0.06      0.90  0.07  0.85  0.21  0.27  
            10      0.30  0.36      1.00  0.05  1.00  1.00  1.00  
            100     1.00  1.00      1.00  0.05  1.00  1.00  1.00  
      0.7   2       0.07  0.07      0.65  0.04  0.65  0.11  0.15  
            10      0.97  0.99      1.00  0.04  1.00  1.00  1.00  
            100     1.00  1.00      1.00  0.04  1.00  1.00  1.00

## Decline regime: final analysis for the paper (Section 3.3)

Fixed design from the parameter exploration above: stable phase at 100 for the first 50 % of the series, then exponential decline to `100 / factor` at the last step. Main case: factor 10, relative noise 0.15. Sensitivity: factor 100. Stress test: relative noise 0.30. Factor 50 is only used for the depth-resolution AUC (50 vs 100). Cell C uses its own random generator; Cells A and B are not affected. Cell C produces the numbers of Tables 4 and 5 of the paper and the AUC values for the depth of the decline reported in the text of Section 3.3.

In [11]:
# ------------------------------------------------------------------
# Cell C: final decline analysis for Section 3.3
# Requires Cells A and B (gen_decline, measures_with_d, draw_normal_pos, ...).
# ------------------------------------------------------------------
PAPER_NOISES  = [0.15, 0.30]        # main case, stress test (relative SD of the scatter)
PAPER_ONSET   = 0.5                 # decline starts after 50 % of the series
PAPER_FACTORS = [10, 50, 100]       # 10: main case, 100: sensitivity, 50: only for the 50-vs-100 AUC
N_PAPER       = 50000
PAPER_MEAS    = ["PV_hi", "PV_lo", "SDL", "CV", "D"]
rng_paper     = np.random.default_rng(2027)

runs, rows, arows = {}, [], []
for noise in PAPER_NOISES:
    runs[(noise, "stable")] = pd.DataFrame([measures_with_d(draw_normal_pos(rng_paper, 100, 100 * noise, N_STEPS))
                                            for _ in range(N_PAPER)])
    for F in PAPER_FACTORS:
        runs[(noise, F)] = pd.DataFrame([measures_with_d(gen_decline(rng_paper, F, PAPER_ONSET, noise))
                                         for _ in range(N_PAPER)])
    ref_mean = runs[(noise, "stable")][PAPER_MEAS].mean()
    for scen in ["stable"] + PAPER_FACTORS:
        df = runs[(noise, scen)]
        for m in PAPER_MEAS:
            rows.append(dict(noise=noise, scenario=scen, measure=m, mean=df[m].mean(), sd=df[m].std(),
                             response=df[m].mean() / ref_mean[m], pv_lo_mean=df["pv_lo"].mean()))
    for f1, f2 in [(10, 100), (50, 100)]:
        r = dict(noise=noise, pair=f"{f1} vs {f2}")
        for m in PAPER_MEAS:
            r[m] = auc(runs[(noise, f1)][m], runs[(noise, f2)][m])
        arows.append(r)

paper_end = pd.DataFrame(rows)
paper_end.to_csv(OUT_DIR / "decline_paper_end.csv", index=False)
print("End of the series (n = %d), mean and SD over %d series, response = mean / mean of the stable series:" % (N_STEPS, N_PAPER))
display(paper_end.pivot_table(index=["noise", "scenario"], columns="measure", values=["mean", "sd", "response"])
        .loc[:, (slice(None), PAPER_MEAS)].round(3))
paper_auc = pd.DataFrame(arows).set_index(["noise", "pair"])
paper_auc.to_csv(OUT_DIR / "decline_paper_auc.csv")
print("\nDepth resolution (AUC, deeper vs shallower decline):")
display(paper_auc.round(3))

# expanding window: detection power and mean index at selected prefix lengths
PD_FACTORS = [10, 100]
PD_DELAYS  = [10, 20, 30]
PD_KS      = list(range(10, N_STEPS + 1, 2))
PD_SHOW    = [30, 50, 60, 70, 80, 100]          # prefix lengths for the mean-index trajectory
N_PD_REF, N_PD = 5000, 2000
t0 = int(round(PAPER_ONSET * N_STEPS))

prow, traj = [], []
for noise in PAPER_NOISES:
    ref = {m: np.empty((N_PD_REF, len(PD_KS))) for m in PAPER_MEAS}
    for i in range(N_PD_REF):
        z = draw_normal_pos(rng_paper, 100, 100 * noise, N_STEPS)
        for j, k in enumerate(PD_KS):
            r = measures_with_d(z[:k])
            for m in PAPER_MEAS:
                ref[m][i, j] = r[m]
    thr = {m: np.quantile(ref[m], 0.95, axis=0) for m in PAPER_MEAS}
    for m in PAPER_MEAS:
        for k in PD_SHOW:
            traj.append(dict(noise=noise, factor="stable", measure=m, k=k, mean=ref[m][:, PD_KS.index(k)].mean()))
    for F in PD_FACTORS:
        hit  = {m: np.zeros((N_PD, len(PD_KS)), dtype=bool) for m in PAPER_MEAS}
        vals = {m: np.zeros((N_PD, len(PD_KS))) for m in PAPER_MEAS}
        for i in range(N_PD):
            z = gen_decline(rng_paper, F, PAPER_ONSET, noise)
            for j, k in enumerate(PD_KS):
                r = measures_with_d(z[:k])
                for m in PAPER_MEAS:
                    vals[m][i, j] = r[m]
                    hit[m][i, j] = r[m] > thr[m][j]
        for m in PAPER_MEAS:
            row = dict(noise=noise, factor=F, measure=m)
            for d in PD_DELAYS:
                row[f"power +{d}"] = hit[m][:, PD_KS.index(t0 + d)].mean()
            row["power end"] = hit[m][:, -1].mean()
            prow.append(row)
            for k in PD_SHOW:
                traj.append(dict(noise=noise, factor=F, measure=m, k=k, mean=vals[m][:, PD_KS.index(k)].mean()))
paper_det = pd.DataFrame(prow)
paper_det.to_csv(OUT_DIR / "decline_paper_detection.csv", index=False)
paper_traj = pd.DataFrame(traj)
paper_traj.to_csv(OUT_DIR / "decline_paper_trajectory.csv", index=False)
print("\nShare of decline series above the 95 % threshold of stable series of the same length (d steps after the onset):")
display(paper_det.pivot_table(index=["noise", "factor"], columns="measure",
                              values=["power +10", "power +20", "power +30", "power end"])
        .loc[:, (slice(None), PAPER_MEAS)].round(2))
print("\nMean index at prefix length k (stable = stable reference series):")
display(paper_traj.pivot_table(index=["noise", "factor", "k"], columns="measure", values="mean")[PAPER_MEAS].round(3))

End of the series (n = 100), mean and SD over 50000 series, response = mean / mean of the stable series:


mean response     sd   mean response     sd   mean response  \
measure         PV_hi    PV_hi  PV_hi  PV_lo    PV_lo  PV_lo    SDL      SDL   
noise scenario                                                                 
0.15  10        0.469    3.099  0.006  0.669    4.019  0.009  0.771    5.001   
      50        0.576    3.803  0.005  0.884    5.311  0.005  1.296    8.400   
      100       0.604    3.992  0.004  0.932    5.601  0.003  1.530    9.920   
      stable    0.151    1.000  0.010  0.166    1.000  0.012  0.154    1.000   
0.30  10        0.508    1.821  0.010  0.709    1.979  0.024  0.833    2.380   
      50        0.608    2.182  0.008  0.894    2.496  0.009  1.331    3.802   
      100       0.635    2.279  0.007  0.935    2.608  0.005  1.547    4.420   
      stable    0.279    1.000  0.019  0.358    1.000  0.056  0.350    1.000   

                   sd   mean response     sd   mean response     sd  
measure           SDL     CV       CV     CV      D        D      D  
noise scenario                                                       
0.15  10        0.016  0.541    3.614  0.009  0.175    1.011  0.016  
      50        0.017  0.699    4.670  0.007  0.178    1.032  0.017  
      100       0.019  0.743    4.961  0.007  0.180    1.041  0.018  
      stable    0.012  0.150    1.000  0.011  0.173    1.000  0.016  
0.30  10        0.038  0.614    2.056  0.020  0.373    1.002  0.042  
      50        0.036  0.765    2.562  0.019  0.374    1.003  0.042  
      100       0.033  0.808    2.704  0.018  0.368    0.989  0.042  
      stable    0.050  0.299    1.000  0.023  0.372    1.000  0.043


Depth resolution (AUC, deeper vs shallower decline):


PV_hi  PV_lo  SDL     CV      D
noise pair                                      
0.15  10 vs 100  1.000    1.0  1.0  1.000  0.582
      50 vs 100  1.000    1.0  1.0  1.000  0.524
0.30  10 vs 100  1.000    1.0  1.0  1.000  0.470
      50 vs 100  0.995    1.0  1.0  0.949  0.465


Share of decline series above the 95 % threshold of stable series of the same length (d steps after the onset):


power +10 power +20 power +30 power end power +10 power +20  \
measure          PV_hi     PV_hi     PV_hi     PV_hi     PV_lo     PV_lo   
noise factor                                                               
0.15  10          0.64      1.00       1.0       1.0      0.66      1.00   
      100         1.00      1.00       1.0       1.0      1.00      1.00   
0.30  10          0.13      0.85       1.0       1.0      0.06      0.27   
      100         0.50      1.00       1.0       1.0      0.14      1.00   

             power +30 power end power +10 power +20 power +30 power end  \
measure          PV_lo     PV_lo       SDL       SDL       SDL       SDL   
noise factor                                                               
0.15  10          1.00       1.0      0.67      1.00       1.0       1.0   
      100         1.00       1.0      1.00      1.00       1.0       1.0   
0.30  10          0.99       1.0      0.07      0.32       1.0       1.0   
      100         1.00       1.0      0.18      1.00       1.0       1.0   

             power +10 power +20 power +30 power end power +10 power +20  \
measure             CV        CV        CV        CV         D         D   
noise factor                                                               
0.15  10          0.63      1.00       1.0       1.0      0.06      0.07   
      100         1.00      1.00       1.0       1.0      0.06      0.07   
0.30  10          0.14      0.85       1.0       1.0      0.06      0.06   
      100         0.49      1.00       1.0       1.0      0.05      0.05   

             power +30 power end  
measure              D         D  
noise factor                      
0.15  10          0.06      0.07  
      100         0.08      0.13  
0.30  10          0.06      0.06  
      100         0.05      0.04


Mean index at prefix length k (stable = stable reference series):


measure           PV_hi  PV_lo    SDL     CV      D
noise factor k                                     
0.15  10     30   0.152  0.166  0.153  0.149  0.173
             50   0.152  0.167  0.154  0.150  0.173
             60   0.178  0.201  0.189  0.178  0.174
             70   0.251  0.308  0.304  0.259  0.174
             80   0.332  0.440  0.452  0.355  0.174
             100  0.469  0.669  0.772  0.541  0.175
      100    30   0.150  0.165  0.152  0.148  0.171
             50   0.151  0.166  0.154  0.149  0.172
             60   0.219  0.270  0.268  0.226  0.175
             70   0.344  0.514  0.546  0.373  0.177
             80   0.453  0.727  0.865  0.513  0.178
             100  0.605  0.932  1.531  0.743  0.180
      stable 30   0.152  0.166  0.153  0.149  0.173
             50   0.151  0.166  0.154  0.150  0.173
             60   0.151  0.166  0.154  0.150  0.173
             70   0.151  0.166  0.154  0.150  0.173
             80   0.151  0.166  0.154  0.150  0.173
             100  0.151  0.166  0.154  0.150  0.173
0.30  10     30   0.278  0.347  0.340  0.297  0.371
             50   0.278  0.352  0.345  0.297  0.371
             60   0.291  0.370  0.364  0.314  0.371
             70   0.336  0.435  0.437  0.370  0.372
             80   0.395  0.528  0.551  0.446  0.372
             100  0.507  0.708  0.831  0.613  0.372
      100    30   0.278  0.349  0.342  0.297  0.371
             50   0.278  0.353  0.346  0.297  0.371
             60   0.318  0.412  0.411  0.345  0.373
             70   0.413  0.587  0.630  0.462  0.374
             80   0.504  0.758  0.919  0.587  0.374
             100  0.635  0.935  1.548  0.807  0.368
      stable 30   0.278  0.349  0.343  0.297  0.372
             50   0.278  0.352  0.345  0.298  0.371
             60   0.278  0.353  0.346  0.298  0.370
             70   0.278  0.354  0.347  0.298  0.370
             80   0.278  0.355  0.347  0.298  0.371
             100  0.278  0.357  0.349  0.298  0.372

## Can crashes be told from ordinary fluctuations of unknown amplitude?

Question raised during the revision: a larger *response factor* of an index (e.g. 10-fold instead of 4-fold in the Rare regime) is a statement about its scale, not about its ability to separate crashes from ordinary fluctuations. Here the separation is measured directly. Ordinary series: fluctuations of unknown amplitude (Gaussian with relative SD ~ U(0.05, 0.30), or log-normal with log-SD ~ U(0.05, 0.50)). Crash series: the same family and amplitude distribution plus five crashes at random positions to `100 / f` (relative SD 0.5, as in the Rare regime). AUC = probability that a crash series has a higher index value than an ordinary series (0.5 = no separation). Exploratory. The Gaussian version with random amplitude is repeated, together with known-amplitude controls, in Cell F (Table 6 of the paper); the log-normal variant is not used in the paper.

In [12]:
# ------------------------------------------------------------------
# Cell D: discrimination of crash series from ordinary fluctuations of unknown amplitude
# ------------------------------------------------------------------
D_N       = 10000
D_FACTORS = [2, 5, 10, 100]                 # crash depth: abundance during crashes = 100 / f
D_N_CRASH = 5
D_MEAS    = ["PV_hi", "PV_lo", "SDL", "CV", "D"]
rng_d     = np.random.default_rng(8080)

def ordinary_gauss(rng):
    s = rng.uniform(0.05, 0.30)
    return draw_normal_pos(rng, 100, 100 * s, N_STEPS)

def ordinary_lognorm(rng):
    s = rng.uniform(0.05, 0.50)
    return np.maximum(np.round(100 * np.exp(rng.normal(0, s, N_STEPS))), 1)

def with_crashes(rng, base, f):
    z = base(rng).copy()
    idx = rng.choice(N_STEPS, D_N_CRASH, replace=False)
    z[idx] = draw_around_path(rng, np.full(D_N_CRASH, 100.0 / f), 0.5)
    return z

drows = []
for fam, base in [("Gaussian, rel. SD 0.05-0.30", ordinary_gauss), ("log-normal, log-SD 0.05-0.50", ordinary_lognorm)]:
    ordv = pd.DataFrame([measures_with_d(base(rng_d)) for _ in range(D_N)])
    for f in D_FACTORS:
        cr = pd.DataFrame([measures_with_d(with_crashes(rng_d, base, f)) for _ in range(D_N)])
        r = dict(family=fam, crash_factor=f)
        for m in D_MEAS:
            r[m] = auc(ordv[m], cr[m])
        drows.append(r)
crash_auc = pd.DataFrame(drows).set_index(["family", "crash_factor"])
crash_auc.to_csv(OUT_DIR / "crash_vs_ordinary_auc.csv")
print("AUC: crash series (5 crashes to 100/f) vs ordinary fluctuations of unknown amplitude (%d series per class):" % D_N)
display(crash_auc.round(3))

AUC: crash series (5 crashes to 100/f) vs ordinary fluctuations of unknown amplitude (10000 series per class):


PV_hi  PV_lo    SDL     CV      D
family                       crash_factor                                   
Gaussian, rel. SD 0.05-0.30  2             0.633  0.755  0.796  0.665  0.677
                             5             0.741  0.976  0.989  0.814  0.863
                             10            0.771  0.999  1.000  0.867  0.946
                             100           0.800  1.000  1.000  0.914  1.000
log-normal, log-SD 0.05-0.50 2             0.573  0.656  0.671  0.569  0.598
                             5             0.652  0.887  0.900  0.639  0.735
                             10            0.678  0.993  0.993  0.667  0.821
                             100           0.702  1.000  1.000  0.695  0.977

## Data for the trajectory figure (decline regime, expanding window)

Two quantities per prefix length k for the decline regime with f = 10 and onset after 50 steps, at relative SD 0.15 (main case) and 0.30 (stress test): the mean response (mean index at length k divided by the mean of the same index in stable series of length k; Fig. panel a) and the detection power (share of decline series above the 95th percentile of the same index in stable series of length k; Fig. panel b). Both are written to `decline_trajectory_fig.csv` for all k (step 2); the cell displays them for selected k only.

In [13]:
# ------------------------------------------------------------------
# Cell E: trajectory data for the figure (written to decline_trajectory_fig.csv)
# ------------------------------------------------------------------
TR_NOISES, TR_F, TR_ONSET = [0.15, 0.30], 10, 0.5
TR_KS   = list(range(10, N_STEPS + 1, 2))
TR_MEAS = ["PV_hi", "PV_lo", "SDL", "CV", "D"]
N_TR_REF, N_TR = 5000, 2000
rng_tr = np.random.default_rng(6060)

trows = []
for noise in TR_NOISES:
    ref = {m: np.empty((N_TR_REF, len(TR_KS))) for m in TR_MEAS}
    for i in range(N_TR_REF):
        z = draw_normal_pos(rng_tr, 100, 100 * noise, N_STEPS)
        for j, k in enumerate(TR_KS):
            r = measures_with_d(z[:k])
            for m in TR_MEAS:
                ref[m][i, j] = r[m]
    thr = {m: np.quantile(ref[m], 0.95, axis=0) for m in TR_MEAS}
    vals = {m: np.empty((N_TR, len(TR_KS))) for m in TR_MEAS}
    for i in range(N_TR):
        z = gen_decline(rng_tr, TR_F, TR_ONSET, noise)
        for j, k in enumerate(TR_KS):
            r = measures_with_d(z[:k])
            for m in TR_MEAS:
                vals[m][i, j] = r[m]
    for m in TR_MEAS:
        for j, k in enumerate(TR_KS):
            trows.append(dict(noise=noise, measure=m, k=k,
                              mean_stable=ref[m][:, j].mean(), mean_decline=vals[m][:, j].mean(),
                              response=vals[m][:, j].mean() / ref[m][:, j].mean(),
                              power=(vals[m][:, j] > thr[m][j]).mean()))
traj_fig = pd.DataFrame(trows)
traj_fig.to_csv(OUT_DIR / "decline_trajectory_fig.csv", index=False)
print("saved", len(traj_fig), "rows")
SHOW_K = [50, 60, 70, 80, 100]
print("\nMean response (mean index at length k / mean index of stable series of length k):")
display(traj_fig[traj_fig.k.isin(SHOW_K)].pivot_table(
    index=["noise", "k"], columns="measure", values="response")[TR_MEAS].round(2))
print("\nDetection power (share of decline series above the 95th percentile of stable series of length k; nominal 0.05):")
display(traj_fig[traj_fig.k.isin(SHOW_K)].pivot_table(
    index=["noise", "k"], columns="measure", values="power")[TR_MEAS].round(2))

saved 460 rows

Mean response (mean index at length k / mean index of stable series of length k):


measure    PV_hi  PV_lo   SDL    CV     D
noise k                                  
0.15  50    1.00   1.00  1.00  1.00  1.00
      60    1.18   1.21  1.23  1.19  1.00
      70    1.66   1.85  1.97  1.73  1.00
      80    2.19   2.65  2.93  2.37  1.01
      100   3.10   4.01  4.99  3.61  1.01
0.30  50    1.00   1.00  1.00  1.00  1.00
      60    1.04   1.05  1.05  1.05  1.00
      70    1.21   1.22  1.25  1.24  1.00
      80    1.42   1.48  1.58  1.50  1.00
      100   1.82   1.98  2.38  2.06  1.00


Detection power (share of decline series above the 95th percentile of stable series of length k; nominal 0.05):


measure    PV_hi  PV_lo   SDL    CV     D
noise k                                  
0.15  50    0.06   0.05  0.06  0.06  0.05
      60    0.66   0.68  0.68  0.64  0.06
      70    1.00   1.00  1.00  1.00  0.06
      80    1.00   1.00  1.00  1.00  0.06
      100   1.00   1.00  1.00  1.00  0.07
0.30  50    0.05   0.05  0.05  0.05  0.05
      60    0.12   0.07  0.07  0.12  0.05
      70    0.82   0.25  0.31  0.82  0.06
      80    1.00   0.99  1.00  1.00  0.05
      100   1.00   1.00  1.00  1.00  0.05

## Controls: why do the crash analysis (Cell D) and the decline detection (Cells B, C, E) rank the indices differently?

Two design differences: (1) in Cell D the amplitude of the ordinary fluctuations is **unknown** (drawn at random per series), in the decline detection the threshold is calibrated on stable series with the **same, known** amplitude; (2) a crash series contains isolated lows far below anything ordinary fluctuations produce, a decline moves the bulk of the observations. Control 1 repeats the crash comparison with a known (fixed) amplitude and with different ranges of unknown amplitude. Control 2 repeats the decline comparison (as AUC against stable series of the same length) with known and unknown amplitude, at series length 70 (20 steps after the onset) and 100.

Cell F produces Table 6 of the paper (crash comparison) and the decline AUC values that Section 3.3 mentions as "not shown".

In [14]:
# ------------------------------------------------------------------
# Cell F: controls (known vs unknown amplitude; crash vs decline)
# ------------------------------------------------------------------
N_CTRL   = 5000
CTRL_MEAS = ["PV_hi", "PV_lo", "SDL", "CV", "D"]
rng_c    = np.random.default_rng(1234)

def gauss_series(s):
    return draw_normal_pos(rng_c, 100, 100 * s, N_STEPS)

def add_crashes(z, f, n_crash=5):
    z = z.copy()
    idx = rng_c.choice(N_STEPS, n_crash, replace=False)
    z[idx] = draw_around_path(rng_c, np.full(n_crash, 100.0 / f), 0.5)
    return z

# Control 1: crash series vs ordinary series; amplitude known (fixed) or unknown (random per series)
amp_settings = [("fixed s = 0.05", lambda: 0.05), ("fixed s = 0.15", lambda: 0.15), ("fixed s = 0.30", lambda: 0.30),
                ("random s ~ U(0.14, 0.16)", lambda: rng_c.uniform(0.14, 0.16)),
                ("random s ~ U(0.05, 0.30)", lambda: rng_c.uniform(0.05, 0.30)),
                ("random s ~ U(0.02, 0.50)", lambda: rng_c.uniform(0.02, 0.50))]
c1 = []
for label, sfun in amp_settings:
    ordv = pd.DataFrame([measures_with_d(gauss_series(sfun())) for _ in range(N_CTRL)])
    for f in [2, 5, 10]:
        cr = pd.DataFrame([measures_with_d(add_crashes(gauss_series(sfun()), f)) for _ in range(N_CTRL)])
        c1.append(dict(amplitude=label, crash_factor=f, **{m: auc(ordv[m], cr[m]) for m in CTRL_MEAS}))
ctrl_crash = pd.DataFrame(c1).set_index(["amplitude", "crash_factor"])
ctrl_crash.to_csv(OUT_DIR / "control_crash_auc.csv")
print("Control 1: AUC, 5 crashes to 100/f vs ordinary Gaussian fluctuations")
display(ctrl_crash.round(3))

# Control 2: decline series vs stable series of the same length; amplitude known or unknown
c2 = []
for label, sfun in [("fixed s = 0.15", lambda: 0.15), ("fixed s = 0.30", lambda: 0.30),
                    ("random s ~ U(0.05, 0.30)", lambda: rng_c.uniform(0.05, 0.30))]:
    stable = [gauss_series(sfun()) for _ in range(N_CTRL)]
    for k in [70, 100]:
        ref = pd.DataFrame([measures_with_d(z[:k]) for z in stable])
        for F in [2, 10]:
            dd = pd.DataFrame([measures_with_d(gen_decline(rng_c, F, 0.5, sfun())[:k]) for _ in range(N_CTRL)])
            c2.append(dict(amplitude=label, length=k, factor=F, **{m: auc(ref[m], dd[m]) for m in CTRL_MEAS}))
ctrl_decline = pd.DataFrame(c2).set_index(["amplitude", "length", "factor"])
ctrl_decline.to_csv(OUT_DIR / "control_decline_auc.csv")
print("\nControl 2: AUC, decline (onset after 50 steps, to 100/f) vs stable series of the same length")
display(ctrl_decline.round(3))

Control 1: AUC, 5 crashes to 100/f vs ordinary Gaussian fluctuations


PV_hi  PV_lo    SDL     CV      D
amplitude                crash_factor                                   
fixed s = 0.05           2             1.000  1.000  1.000  1.000  1.000
                         5             1.000  1.000  1.000  1.000  1.000
                         10            1.000  1.000  1.000  1.000  1.000
fixed s = 0.15           2             0.974  0.990  0.994  0.984  0.972
                         5             1.000  1.000  1.000  1.000  1.000
                         10            1.000  1.000  1.000  1.000  1.000
fixed s = 0.30           2             0.773  0.766  0.769  0.768  0.752
                         5             0.967  0.966  0.978  0.964  0.973
                         10            0.989  0.991  0.999  0.990  0.997
random s ~ U(0.14, 0.16) 2             0.963  0.987  0.991  0.977  0.964
                         5             1.000  1.000  1.000  1.000  1.000
                         10            1.000  1.000  1.000  1.000  1.000
random s ~ U(0.05, 0.30) 2             0.625  0.746  0.788  0.659  0.668
                         5             0.730  0.972  0.986  0.807  0.854
                         10            0.766  0.998  0.999  0.862  0.944
random s ~ U(0.02, 0.50) 2             0.557  0.610  0.625  0.572  0.573
                         5             0.633  0.783  0.796  0.657  0.690
                         10            0.661  0.907  0.912  0.692  0.768


Control 2: AUC, decline (onset after 50 steps, to 100/f) vs stable series of the same length


PV_hi  PV_lo    SDL     CV      D
amplitude                length factor                                   
fixed s = 0.15           70     2       0.828  0.825  0.818  0.827  0.502
                                10      1.000  1.000  1.000  1.000  0.524
                         100    2       1.000  1.000  1.000  1.000  0.508
                                10      1.000  1.000  1.000  1.000  0.542
fixed s = 0.30           70     2       0.587  0.558  0.554  0.598  0.503
                                10      0.969  0.887  0.888  0.969  0.508
                         100    2       0.970  0.862  0.860  0.980  0.488
                                10      1.000  0.997  1.000  1.000  0.505
random s ~ U(0.05, 0.30) 70     2       0.570  0.568  0.568  0.570  0.500
                                10      0.873  0.892  0.907  0.870  0.500
                         100    2       0.873  0.854  0.850  0.861  0.502
                                10      1.000  1.000  1.000  1.000  0.505

## Cauchy regime: how often do extreme events occur, and do booms or crashes influence the indices more?

Cell G regenerates the 50,000 Cauchy series of the main simulation (same seed; the generator is advanced past the Normal regime) and counts extreme events: a *crash* is an abundance of at most 100/f, a *boom* an abundance of at least 100·f (f = 5, 10, 100; location 100). It reports (1) the frequency of events per series, (2) the index values by event class, (3) the effect of replacing booms or crashes by the location value, and (4) the share of the pairwise sums (pv<sub>lo</sub>, PV<sub>hi</sub>) that comes from pairs containing a boom or a crash. Used for the description of the Cauchy regime in Section 3.2. Runtime about 40 s.

In [15]:
# ------------------------------------------------------------------
# Cell G: Cauchy regime - frequency of extreme events (booms and crashes) and their influence
# Requires the cells above (all_measures, gen_normal, SEED, N_RUNS, N_STEPS, OUT_DIR, np, pd).
# The 50,000 Cauchy series of the main simulation are regenerated exactly: the main simulation
# draws the Normal series first, so the generator is advanced by N_RUNS Normal series.
# (SEED, N_RUNS and the order of CONDITIONS must be unchanged.)
# Event = deviation from the location (100) by at least a factor f: crash z <= 100/f, boom z >= 100*f.
# ------------------------------------------------------------------
rng_cy = np.random.default_rng(SEED)
for _ in range(N_RUNS):
    gen_normal(rng_cy)
cy_x = np.round(100 + 5 * np.stack([rng_cy.standard_cauchy(N_STEPS) for _ in range(N_RUNS)]))
cy_z = np.maximum(np.abs(cy_x), 1)                      # same transformation as gen_cauchy()
cy_m = pd.DataFrame([all_measures(z) for z in cy_z])
COLS = ["PV_hi", "PV_lo", "SDL", "CV", "pv_lo"]

# 1) How often do events occur? (factor 5, 10, 100)
rows = []
for f in (5, 10, 100):
    nc, nb = (cy_z <= 100 / f).sum(1), (cy_z >= 100 * f).sum(1)
    rows.append(dict(factor=f, crashes_per_run=nc.mean(), booms_per_run=nb.mean(),
                     no_event=np.mean(nc + nb == 0), one_event=np.mean(nc + nb == 1), two_plus=np.mean(nc + nb >= 2),
                     any_boom=np.mean(nb >= 1), any_crash=np.mean(nc >= 1), both=np.mean((nb >= 1) & (nc >= 1)),
                     two_plus_booms=np.mean(nb >= 2), two_plus_crashes=np.mean(nc >= 2)))
cy_freq = pd.DataFrame(rows).set_index("factor")
cy_freq.to_csv(OUT_DIR / "cauchy_event_frequency.csv")
print("Frequency of extreme events per Cauchy series (n = %d steps):" % N_STEPS)
display(cy_freq.round(3))
print("Share of booms >= 1000 that stem from negative raw draws (reflected by abs): %.2f"
      % ((cy_x <= -1000).sum() / (cy_z >= 1000).sum()))

# 2) Index values by event class (factor 10)
crash, boom = cy_z <= 10, cy_z >= 1000
nc, nb = crash.sum(1), boom.sum(1)
cls = np.select([(nb == 0) & (nc == 0), (nb == 1) & (nc == 0), (nb == 0) & (nc == 1),
                 (nb >= 2) & (nc == 0), (nb == 0) & (nc >= 2)],
                ["none", "1 boom", "1 crash", ">=2 booms", ">=2 crashes"], "boom + crash")
order = ["none", "1 boom", "1 crash", ">=2 booms", ">=2 crashes", "boom + crash"]
by_class = cy_m.assign(cls=cls).groupby("cls")[COLS].mean().loc[order]
by_class.insert(0, "share_of_runs", pd.Series(cls).value_counts(normalize=True).reindex(order).values)
by_class.to_csv(OUT_DIR / "cauchy_by_event_class.csv")
print("\nMean index by event class (factor 10):")
display(by_class.round(3))

# 3) Influence of booms vs crashes: replace them by the location (100) and recompute
def _means(mask):
    zc = cy_z.copy(); zc[mask] = 100.0
    return pd.DataFrame([all_measures(z) for z in zc])[COLS].mean()
influence = pd.DataFrame({"original": cy_m[COLS].mean(), "booms -> 100": _means(boom),
                          "crashes -> 100": _means(crash), "both -> 100": _means(boom | crash)})
influence.to_csv(OUT_DIR / "cauchy_boom_vs_crash_influence.csv")
print("\nMean index when booms / crashes (factor 10) are replaced by 100:")
display(influence.round(3))

# 4) Share of the pairwise sum (pv_lo, PV_hi) that comes from pairs containing a boom / a crash
I, J = np.triu_indices(N_STEPS, 1)
S_lo, S_hi = np.zeros((N_RUNS, 4)), np.zeros((N_RUNS, 4))   # bulk-bulk, boom, crash, boom+crash
for s in range(0, N_RUNS, 2000):
    a, b = cy_z[s:s + 2000, I], cy_z[s:s + 2000, J]
    hb = boom[s:s + 2000, I] | boom[s:s + 2000, J]
    hc = crash[s:s + 2000, I] | crash[s:s + 2000, J]
    k = np.select([hb & ~hc, hc & ~hb, hb & hc], [1, 2, 3], 0)
    for name, d, S in (("lo", np.abs(a - b) / np.minimum(a, b), S_lo), ("hi", np.abs(a - b) / np.maximum(a, b), S_hi)):
        for c in range(4):
            S[s:s + 2000, c] = (d * (k == c)).sum(1)
share = pd.DataFrame({"pv_lo": S_lo.sum(0) / S_lo.sum(), "PV_hi": S_hi.sum(0) / S_hi.sum()},
                     index=["no extreme", "pair with boom", "pair with crash", "boom-crash pair"])
share.to_csv(OUT_DIR / "cauchy_pair_shares.csv")
print("\nShare of the pooled pairwise sums by pair type (all series pooled):")
display(share.round(3))
big = cy_z.max(1) >= 10000
print("Series with a boom >= 100-fold: %.3f of all series, %.3f of the pooled pv_lo sum"
      % (big.mean(), S_lo[big].sum() / S_lo.sum()))


Frequency of extreme events per Cauchy series (n = 100 steps):


,crashes_per_run,booms_per_run,no_event,one_event,two_plus,any_boom,any_crash,both,two_plus_booms,two_plus_crashes
factor,,,,,,,,,,
5,0.680,0.661,0.258,0.353,0.389,0.485,0.495,0.238,0.143,0.149
10,0.339,0.321,0.516,0.342,0.142,0.274,0.288,0.078,0.042,0.046
100,0.049,0.032,0.922,0.075,0.003,0.032,0.048,0.001,0.000,0.001


Share of booms >= 1000 that stem from negative raw draws (reflected by abs): 0.45

Mean index by event class (factor 10):


,share_of_runs,PV_hi,PV_lo,SDL,CV,pv_lo
cls,,,,,,
none,0.516,0.161,0.252,0.271,0.382,0.347
1 boom,0.166,0.176,0.478,0.433,2.310,3.025
1 crash,0.176,0.176,0.461,0.421,0.397,1.001
>=2 booms,0.030,0.193,0.619,0.559,3.196,3.837
>=2 crashes,0.034,0.193,0.595,0.541,0.418,1.738
boom + crash,0.078,0.197,0.643,0.574,2.442,4.008



Mean index when booms / crashes (factor 10) are replaced by 100:


,original,booms -> 100,crashes -> 100,both -> 100
PV_hi,0.171,0.165,0.165,0.160
PV_lo,0.379,0.318,0.319,0.250
SDL,0.366,0.319,0.320,0.270
CV,0.951,0.386,0.945,0.380
pv_lo,1.344,0.568,1.071,0.344



Share of the pooled pairwise sums by pair type (all series pooled):


,pv_lo,PV_hi
no extreme,0.254,0.927
pair with boom,0.540,0.035
pair with crash,0.167,0.037
boom-crash pair,0.039,0.000


Series with a boom >= 100-fold: 0.032 of all series, 0.475 of the pooled pv_lo sum


## SDL versus PV<sub>lo</sub>: is SDL always better in separating crashes and declines from ordinary fluctuations?

Section 3.3 and the Discussion state that SDL is not uniformly better than PV<sub>lo</sub>. Cells H and I reproduce this comparison, including the additional runs that are not shown as a table in the paper. Both compare the two indices by the AUC (probability that the event series has the higher index value than the reference series; rank-based, so the standardization of PV<sub>lo</sub> and the choice of DDOF do not matter) and give the SE of the AUC difference by a bootstrap. A difference of more than 3 SE counts as a difference; scenarios in which both AUC values exceed 0.9995 are marked as *ceiling* and are not informative.

- **Cell H**: series with 1, 5 or 15 crashes (to 100/f, f = 2, 5, 10, 100) versus ordinary Gaussian fluctuations; known baseline (relative SD 0.05, 0.15, 0.30, 0.50) and unknown baseline (three ranges of the relative SD). The first part of the comparison corresponds to Table 6 (Cell F), the rest extends it.
- **Cell I**: decline series (as in Cells A and C, onset after 50 steps, final abundance 100/F) versus stable series of the same length (k = 60 to 100), known and unknown baseline.

Both cells are self-contained apart from the helper functions of the first cells, use their own random number generators (fixed seeds, so that the numbers are identical on every run) and write CSV files to the `output` folder. Cell I needs the helper functions defined in Cell H. Runtime: about 2.5 and 1.5 minutes (reduce `N_CMP` for a quick check).

In [16]:
# ------------------------------------------------------------------
# Cell H: SDL versus PV_lo - separation of crash series from ordinary fluctuations
# Section 3.3: "SDL is not uniformly better" (additional runs beyond Table 6, not shown in the paper).
# Requires the settings cell (imports, N_STEPS, OUT_DIR), the measures cell (pv_indices) and the
# generators cell (draw_normal_pos). Self-contained otherwise; separate random number generators.
#
# Crash series: Gaussian fluctuations around 100 with relative SD s, plus k crashes at random positions
# to 100/f (relative SD of the crashed abundance 0.5, as in Cell F). Ordinary series: the same without crashes.
# AUC = probability that a crash series has a higher index than an ordinary series (ties count 0.5).
# The AUC depends only on ranks (monotone standardizations and the choice of DDOF do not matter).
# SE of the AUC difference: bootstrap of both classes (the same resample for both indices).
# Runtime: about 2.5 minutes with N_CMP = 12000.
# ------------------------------------------------------------------
N_CMP      = 12_000                          # series per class and scenario
CMP_NBOOT  = 150                             # bootstrap replicates for the SE of the AUC difference
CMP_KNOWN  = (0.05, 0.15, 0.30, 0.50)        # known baseline: relative SD of the ordinary fluctuations
CMP_UNKNOWN = ((0.14, 0.16), (0.05, 0.30), (0.02, 0.50))   # unknown baseline: relative SD ~ U(lo, hi) per series
CMP_F      = (2, 5, 10, 100)                 # crash depth: abundance 100/f (f = 100: about 1, the maximal depth of the Cauchy regime)
CMP_K      = (1, 5, 15)                      # number of crashes per series

def _cmp_around(rng, mean, rel_sd):
    x = np.round(rng.normal(mean, rel_sd * mean)); bad = x < 1
    while bad.any():
        x[bad] = np.round(rng.normal(mean[bad], rel_sd * mean[bad])); bad = x < 1
    return x

def _cmp_gauss(rng, s):
    return draw_normal_pos(rng, 100, 100 * s, N_STEPS)

def _cmp_crash(rng, z, f, k):
    z = z.copy(); idx = rng.choice(N_STEPS, k, replace=False)
    z[idx] = _cmp_around(rng, np.full(k, 100.0 / f), 0.5)
    return z

def _cmp_twos(series):                           # columns: SDL, PV_lo
    return np.array([(np.log(z).std(ddof=1), pv_indices(z)[1]) for z in series])

def _cmp_auc(a, b):
    r = stats.rankdata(np.concatenate([a, b])); na = len(a)
    return (r[na:].sum() - len(b) * (len(b) + 1) / 2) / (na * len(b))

def _cmp_auc_diff(A, B, rng, nboot):
    """A: indices of the reference class, B: of the event class (columns SDL, PV_lo).
    Returns AUC_SDL, AUC_PVlo, diff = AUC_SDL - AUC_PVlo, SE of diff."""
    auc_sdl, auc_pv = _cmp_auc(A[:, 0], B[:, 0]), _cmp_auc(A[:, 1], B[:, 1])
    d = []
    for _ in range(nboot):
        ia, ib = rng.integers(0, len(A), len(A)), rng.integers(0, len(B), len(B))
        d.append(_cmp_auc(A[ia, 0], B[ib, 0]) - _cmp_auc(A[ia, 1], B[ib, 1]))
    return auc_sdl, auc_pv, auc_sdl - auc_pv, np.std(d)

def _cmp_winner(df, ceiling=0.9995, z=3):
    """Classify scenarios; scenarios in which both indices are at the ceiling (AUC > 0.9995) are not informative."""
    ceil = (df.AUC_SDL > ceiling) & (df.AUC_PVlo > ceiling)
    t = df["diff"] / df["se"].where(df["se"] > 0, np.nan)
    w = np.select([t > z, t < -z], ["SDL better", "PV_lo better"], "no difference")
    return pd.Series(np.where(ceil, "ceiling (both ~1)", w), index=df.index)

_t0, _rows = time.time(), []
for _s in CMP_KNOWN:
    _label, _sf = f"known, s = {_s}", (lambda: _s)
    _rng, _rb = np.random.default_rng(99 + 1 + int(_s * 1000)), np.random.default_rng(1)
    _ref = _cmp_twos([_cmp_gauss(_rng, _sf()) for _ in range(N_CMP)])
    for _f in CMP_F:
        for _k in CMP_K:
            _ev = _cmp_twos([_cmp_crash(_rng, _cmp_gauss(_rng, _sf()), _f, _k) for _ in range(N_CMP)])
            _rows.append(dict(baseline=_label, rel_sd=_s, f=_f, crashes=_k,
                              **dict(zip(["AUC_SDL", "AUC_PVlo", "diff", "se"], _cmp_auc_diff(_ref, _ev, _rb, CMP_NBOOT)))))
    print(f"{_label}: done ({time.time() - _t0:.0f} s)")
for _lo, _hi in CMP_UNKNOWN:
    _rng, _rb = np.random.default_rng(99 + 2 + int(_lo * 1000)), np.random.default_rng(1)
    _sf = lambda: _rng.uniform(_lo, _hi)
    _ref = _cmp_twos([_cmp_gauss(_rng, _sf()) for _ in range(N_CMP)])
    for _f in CMP_F:
        for _k in CMP_K:
            _ev = _cmp_twos([_cmp_crash(_rng, _cmp_gauss(_rng, _sf()), _f, _k) for _ in range(N_CMP)])
            _rows.append(dict(baseline=f"unknown, s ~ U({_lo}, {_hi})", rel_sd=np.nan, f=_f, crashes=_k,
                              **dict(zip(["AUC_SDL", "AUC_PVlo", "diff", "se"], _cmp_auc_diff(_ref, _ev, _rb, CMP_NBOOT)))))
    print(f"unknown, s ~ U({_lo}, {_hi}): done ({time.time() - _t0:.0f} s)")

cmp_crash = pd.DataFrame(_rows)
cmp_crash["result"] = _cmp_winner(cmp_crash)
cmp_crash.to_csv(OUT_DIR / "crash_sdl_vs_pvlo.csv", index=False)

print("\nAUC of SDL and PV_lo (diff = SDL - PV_lo; negative: PV_lo better):")
display(cmp_crash.round(4))
print("\nNumber of scenarios by result (|diff| > 3 SE counts as a difference):")
display(cmp_crash.groupby([cmp_crash.baseline.str.split(",").str[0], "result"]).size().unstack(fill_value=0))
print("\nScenarios in which PV_lo separates clearly better:")
display(cmp_crash[cmp_crash.result == "PV_lo better"].round(4))
_kn = cmp_crash[cmp_crash.rel_sd <= 0.30]               # known baseline, relative SD up to 0.30
print("Known baseline with s <= 0.30: largest advantage of PV_lo (AUC) = %.4f; of SDL = %.4f"
      % (max(0, -_kn["diff"].min()), max(0, _kn["diff"].max())))


known, s = 0.05: done (20 s)
known, s = 0.15: done (41 s)
known, s = 0.3: done (62 s)
known, s = 0.5: done (84 s)
unknown, s ~ U(0.14, 0.16): done (104 s)
unknown, s ~ U(0.05, 0.3): done (124 s)
unknown, s ~ U(0.02, 0.5): done (144 s)

AUC of SDL and PV_lo (diff = SDL - PV_lo; negative: PV_lo better):


,baseline,rel_sd,f,crashes,AUC_SDL,AUC_PVlo,diff,se,result
0,"known, s = 0.05",0.05,2,1,0.9499,0.9250,0.0250,0.0007,SDL better
1,"known, s = 0.05",0.05,2,5,1.0000,1.0000,0.0000,0.0000,ceiling (both ~1)
2,"known, s = 0.05",0.05,2,15,1.0000,1.0000,0.0000,0.0000,ceiling (both ~1)
3,"known, s = 0.05",0.05,5,1,1.0000,1.0000,0.0000,0.0000,ceiling (both ~1)
4,"known, s = 0.05",0.05,5,5,1.0000,1.0000,0.0000,0.0000,ceiling (both ~1)
...,...,...,...,...,...,...,...,...,...
79,"unknown, s ~ U(0.02, 0.5)",NaN,10,5,0.9144,0.9096,0.0048,0.0005,SDL better
80,"unknown, s ~ U(0.02, 0.5)",NaN,10,15,0.9994,0.9912,0.0082,0.0005,SDL better
81,"unknown, s ~ U(0.02, 0.5)",NaN,100,1,0.8501,0.9605,-0.1104,0.0019,PV_lo better
82,"unknown, s ~ U(0.02, 0.5)",NaN,100,5,1.0000,1.0000,-0.0000,0.0000,ceiling (both ~1)



Number of scenarios by result (|diff| > 3 SE counts as a difference):


result,PV_lo better,SDL better,ceiling (both ~1),no difference
baseline,,,,
known,3,18,25,2
unknown,2,20,14,0



Scenarios in which PV_lo separates clearly better:


,baseline,rel_sd,f,crashes,AUC_SDL,AUC_PVlo,diff,se,result
30,"known, s = 0.3",0.3,10,1,0.8639,0.8679,-0.0041,0.0003,PV_lo better
33,"known, s = 0.3",0.3,100,1,0.9939,0.9952,-0.0013,0.0002,PV_lo better
45,"known, s = 0.5",0.5,100,1,0.8627,0.9218,-0.0591,0.0013,PV_lo better
69,"unknown, s ~ U(0.05, 0.3)",NaN,100,1,0.9975,0.9992,-0.0016,0.0002,PV_lo better
81,"unknown, s ~ U(0.02, 0.5)",NaN,100,1,0.8501,0.9605,-0.1104,0.0019,PV_lo better


Known baseline with s <= 0.30: largest advantage of PV_lo (AUC) = 0.0041; of SDL = 0.0272


In [17]:
# ------------------------------------------------------------------
# Cell I: SDL versus PV_lo - separation of decline series from stable series of the same length
# Section 3.3: known baseline, differences below 0.011; unknown baseline, differences within 0.04.
# Requires Cell H (helper functions _cmp_*) and the cells it requires.
# Decline series: stable at 100 for the first 50 % of the series, then exponential decline to 100/F at the
# last step, Gaussian scatter with relative SD s (as in Cell A). Reference: stable series with relative SD s.
# Both are cut to the length k (k = 60, 70, 80, 100: 10, 20, 30 and 50 steps after the onset).
# Runtime: about 1.5 minutes with N_CMP = 12000.
# ------------------------------------------------------------------
CMP_DECL_F   = (2, 10, 100)                  # final abundance 100/F
CMP_DECL_K   = (60, 70, 80, 100)             # series length
CMP_DECL_SET = ("known, s = 0.15", "known, s = 0.30", "unknown, s ~ U(0.05, 0.30)")

def _cmp_decline(rng, F, onset, s, n=N_STEPS):
    t0 = int(round(onset * n)); path = np.full(n, 100.0); m = n - t0
    path[t0:] = 100.0 * F ** (-np.arange(1, m + 1) / m)
    return _cmp_around(rng, path, s)

_t0, _rows = time.time(), []
for _i, _label in enumerate(CMP_DECL_SET):
    _rng, _rb = np.random.default_rng(500 + _i), np.random.default_rng(2)
    _sf = (lambda: 0.15) if _i == 0 else (lambda: 0.30) if _i == 1 else (lambda: _rng.uniform(0.05, 0.30))
    _stable = [_cmp_gauss(_rng, _sf()) for _ in range(N_CMP)]
    for _F in CMP_DECL_F:
        _dec = [_cmp_decline(_rng, _F, 0.5, _sf()) for _ in range(N_CMP)]
        for _k in CMP_DECL_K:
            _A, _B = _cmp_twos([z[:_k] for z in _stable]), _cmp_twos([z[:_k] for z in _dec])
            _rows.append(dict(setting=_label, F=_F, k=_k,
                              **dict(zip(["AUC_SDL", "AUC_PVlo", "diff", "se"], _cmp_auc_diff(_A, _B, _rb, CMP_NBOOT)))))
    print(f"{_label}: done ({time.time() - _t0:.0f} s)")

cmp_decline = pd.DataFrame(_rows)
cmp_decline["result"] = _cmp_winner(cmp_decline)
cmp_decline.to_csv(OUT_DIR / "decline_sdl_vs_pvlo.csv", index=False)

print("\nAUC of SDL and PV_lo, decline versus stable series of the same length (diff = SDL - PV_lo):")
display(cmp_decline.round(4))
print("\nNumber of scenarios by result:")
display(cmp_decline.groupby([cmp_decline.setting.str.split(",").str[0], "result"]).size().unstack(fill_value=0))
_kd = cmp_decline[cmp_decline.setting.str.startswith("known")]
print("Known baseline: largest |diff| = %.4f; largest advantage of PV_lo = %.4f, of SDL = %.4f"
      % (_kd["diff"].abs().max(), max(0, -_kd["diff"].min()), max(0, _kd["diff"].max())))
print("Known baseline, F = 2: mean diff (SDL - PV_lo) = %.4f (negative: PV_lo better)" % _kd[_kd.F == 2]["diff"].mean())
print("Unknown baseline: largest |diff| = %.4f" % cmp_decline[cmp_decline.setting.str.startswith("unknown")]["diff"].abs().max())


known, s = 0.15: done (20 s)
known, s = 0.30: done (41 s)
unknown, s ~ U(0.05, 0.30): done (62 s)

AUC of SDL and PV_lo, decline versus stable series of the same length (diff = SDL - PV_lo):


,setting,F,k,AUC_SDL,AUC_PVlo,diff,se,result
0,"known, s = 0.15",2,60,0.5629,0.5648,-0.0020,0.0005,PV_lo better
1,"known, s = 0.15",2,70,0.8152,0.8231,-0.0079,0.0005,PV_lo better
2,"known, s = 0.15",2,80,0.9868,0.9895,-0.0027,0.0002,PV_lo better
3,"known, s = 0.15",2,100,1.0000,1.0000,0.0000,0.0000,ceiling (both ~1)
4,"known, s = 0.15",10,60,0.9310,0.9291,0.0019,0.0003,SDL better
5,"known, s = 0.15",10,70,1.0000,1.0000,0.0000,0.0000,ceiling (both ~1)
6,"known, s = 0.15",10,80,1.0000,1.0000,0.0000,0.0000,ceiling (both ~1)
7,"known, s = 0.15",10,100,1.0000,1.0000,0.0000,0.0000,ceiling (both ~1)
8,"known, s = 0.15",100,60,1.0000,1.0000,-0.0000,0.0000,ceiling (both ~1)
9,"known, s = 0.15",100,70,1.0000,1.0000,0.0000,0.0000,ceiling (both ~1)



Number of scenarios by result:


result,PV_lo better,SDL better,ceiling (both ~1),no difference
setting,,,,
known,8,5,9,2
unknown,2,5,3,2


Known baseline: largest |diff| = 0.0103; largest advantage of PV_lo = 0.0079, of SDL = 0.0103
Known baseline, F = 2: mean diff (SDL - PV_lo) = -0.0033 (negative: PV_lo better)
Unknown baseline: largest |diff| = 0.0291
